# ShopEase Demo 2: Structured Extraction with Groq

Turn a raw support email into a consistent ticket. Groq returns a function/tool call whose arguments are checked against a Pydantic schema. This demonstrates structured extraction and validation; no OpenRouter or Claude is used.

For live calls, set `GROQ_API_KEY` before launching Jupyter or add it to a local `Module_3/.env` copied from `.env.example`. Without it, the cells run an offline example through the same Pydantic validation.

In [1]:
import os
from pathlib import Path
from groq import Groq

def find_groq_key():
    if os.getenv("GROQ_API_KEY"):
        return os.environ["GROQ_API_KEY"]
    # Optionally read a local Module_3/.env file; the notebook never stores the key.
    for folder in (Path.cwd(), *Path.cwd().parents):
        env_file = folder / ".env"
        if env_file.is_file():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line.startswith("GROQ_API_KEY="):
                    value = line.split("=", 1)[1].strip().strip("\"'")
                    if value and value != "your_groq_key_here":
                        return value
    return None

MODEL = "openai/gpt-oss-120b"
GROQ_API_KEY = find_groq_key()
LIVE_API = bool(GROQ_API_KEY)
client = Groq(api_key=GROQ_API_KEY) if LIVE_API else None
print("Mode:", "LIVE Groq API" if LIVE_API else "OFFLINE PREVIEW (no API request)")

Mode: LIVE Groq API


## 1. Define the output contract

The Pydantic model lists the fields and allowed labels. The model's JSON schema becomes the function parameter schema supplied to Groq.

In [2]:
from pydantic import BaseModel, Field
from typing import Literal

class SupportTicket(BaseModel):
    customer_name: str = Field(description="Full name of the customer")
    order_id: str = Field(description="Order identifier, for example SE-48213")
    issue_category: Literal["damaged_item", "shipping_delay", "refund_request", "product_defect", "other"]
    sentiment: Literal["positive", "neutral", "negative"]
    requested_action: Literal["refund", "replacement", "store_credit", "status_update", "none"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="One-sentence summary of the issue")

print(SupportTicket.model_json_schema())

{'properties': {'customer_name': {'description': 'Full name of the customer', 'title': 'Customer Name', 'type': 'string'}, 'order_id': {'description': 'Order identifier, for example SE-48213', 'title': 'Order Id', 'type': 'string'}, 'issue_category': {'enum': ['damaged_item', 'shipping_delay', 'refund_request', 'product_defect', 'other'], 'title': 'Issue Category', 'type': 'string'}, 'sentiment': {'enum': ['positive', 'neutral', 'negative'], 'title': 'Sentiment', 'type': 'string'}, 'requested_action': {'enum': ['refund', 'replacement', 'store_credit', 'status_update', 'none'], 'title': 'Requested Action', 'type': 'string'}, 'urgency': {'enum': ['low', 'medium', 'high'], 'title': 'Urgency', 'type': 'string'}, 'summary': {'description': 'One-sentence summary of the issue', 'title': 'Summary', 'type': 'string'}}, 'required': ['customer_name', 'order_id', 'issue_category', 'sentiment', 'requested_action', 'urgency', 'summary'], 'title': 'SupportTicket', 'type': 'object'}


## 2. Select a sample email

In [3]:
EMAIL = """Subject: Refund needed ASAP

Hi, this is Priya Nair. I ordered a pair of running shoes, order #SE-48213,
ten days ago and they arrived with a torn sole. I need them for a race this
weekend. Please refund me or send a replacement immediately."""
print(EMAIL)

Subject: Refund needed ASAP

Hi, this is Priya Nair. I ordered a pair of running shoes, order #SE-48213,
ten days ago and they arrived with a torn sole. I need them for a race this
weekend. Please refund me or send a replacement immediately.


## 3. Ask Groq for the structured ticket and validate it

In [4]:
import json

TOOL = {
    "type": "function",
    "function": {
        "name": "extract_support_ticket",
        "description": "Extract a validated support ticket from the customer email.",
        "parameters": SupportTicket.model_json_schema(),
    },
}

def extract_ticket(email):
    if not LIVE_API:
        # Example payload lets the schema and validation steps run without credentials.
        payload = {
            "customer_name": "Priya Nair", "order_id": "SE-48213",
            "issue_category": "damaged_item", "sentiment": "negative",
            "requested_action": "refund", "urgency": "high",
            "summary": "Running shoes arrived with a torn sole and the customer requests a refund or replacement before a race."
        }
    else:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "Extract the requested fields faithfully from the email. Do not invent missing details."},
                {"role": "user", "content": f"Extract a support ticket from this email:\n{email}"},
            ],
            tools=[TOOL],
            tool_choice={"type": "function", "function": {"name": "extract_support_ticket"}},
            temperature=0,
            max_tokens=500,
        )
        tool_call = response.choices[0].message.tool_calls[0]
        payload = json.loads(tool_call.function.arguments)
    return SupportTicket.model_validate(payload)

ticket = extract_ticket(EMAIL)
print(ticket.model_dump_json(indent=2))

{
  "customer_name": "Priya Nair",
  "order_id": "SE-48213",
  "issue_category": "damaged_item",
  "sentiment": "negative",
  "requested_action": "refund",
  "urgency": "high",
  "summary": "Running shoes arrived with a torn sole; customer requests a refund or replacement urgently."
}


## Discussion

The model proposes the fields; Pydantic validates types and enumerated values before downstream code uses the ticket. Try changing the email and inspect how the fields should change. The offline payload is clearly marked as an example, not a model response.